In [12]:
# Setup para estrutura: .../voidknee/src/{compiler, notebooks, out}
import sys
from pathlib import Path

NB_DIR = Path.cwd()
SRC = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR   # -> .../src
OUT = SRC / "out"
OUT.mkdir(parents=True, exist_ok=True)

# Importa o compilador do pacote 'compiler' dentro de src
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
from compiler.voidknee_compiladorV2 import traduzir

# Helper para gerar .c em src/out
def gerar_c(nome: str, fonte_vk: str) -> str:
    c_code = traduzir(fonte_vk)
    destino = OUT / f"{nome}.c"
    destino.write_text(c_code, encoding="utf-8")
    print(f"Gerado: {destino.resolve()}")
    return c_code

print("SRC =", SRC.resolve())
print("OUT =", OUT.resolve())


SRC = C:\Users\user\Documents\Puc\Compiladores\desenvolvimento compilador\voidknee\src
OUT = C:\Users\user\Documents\Puc\Compiladores\desenvolvimento compilador\voidknee\src\out


# VoidKnee V2 🦵 → Tradutor para C

> Linguagem de brinquedo com pipeline completo (**Tokens → Parser → AST → Semântica → CodeGen em C**).  
> Este notebook é a **apresentação** da V2: explique, demonstre e **rode exemplos**.

---

## O que há de novo na V2 (comparado à V1)
- **Condição e laços 100% funcionais** (`sejoelho/outracoisa`, `enquantoDoi`, `praCada`).
- **Operações numéricas completas** com `inteirao`, `flutuante`, `dobradura` (double), `dorzinha` (char) e `verdadeQueDoi` (bool).
- **Vetores e matrizes** com sintaxe `tipo nome[TAM]` e `tipo nome[LIN][COL]` + operações por índice.
- **Strings** via `dorzona` (vetor de `char`): leitura segura, concatenação com `+` e impressão com `mostraAi`.
- **Operadores aritméticos com booleano**: `verdadeiro`/`falso` participam de expressões (mapeados para `int` 1/0 em C).
- Cuidados com **buffer** ao fazer `entradaAi` de `char` e `string`.


## Recapitulação rápida do vocabulário VoidKnee

### Tipos
| VoidKnee        | Significado            | Mapeamento C             | Observações |
|-----------------|------------------------|--------------------------|-------------|
| `inteirao`      | inteiro                | `int`                    |             |
| `flutuante`     | ponto flutuante        | `float`                  |             |
| `dobradura`     | dupla precisão         | `double`                 |             |
| `verdadeQueDoi` | booleano               | `int` (0/1)              | literais: `verdadeiro`, `falso` |
| `dorzinha`      | caractere              | `char`                   | leitura de char usa **`" %c"`** para ignorar `\n` |
| `dorzona`       | string (char array)    | `char[N]`                | declarar com tamanho: `dorzona nome[30];` |

### Controle de fluxo e E/S
- `sejoelho (cond) { ... } outracoisa { ... }`
- `enquantoDoi (cond) { ... }`
- `praCada (inicial; cond; passo) { ... }`  *(equivale ao `for` do C)*
- `mostraAi(expr);`  *(imprime; aceita concatenação com `+`)*
- `entradaAi(varOuElemento);`  *(lê para variável escalar, elemento de vetor/matriz, ou `dorzona` inteira)*

### Operadores
- Aritméticos: `+`, `-`, `*`, `/`, `%`
- Relacionais: `<`, `<=`, `>`, `>=`, `==`, `!=`
- Lógicos: `&&`, `||`, `!`
- Indexação: `a[i]`, `m[i][j]`
- **Concatenação de strings**: `+` (se qualquer lado for string, o resultado é string)


# Explicação **passo a passo**

> Foco no **pipeline** (Léxico → Parser/AST → Semântica → Geração de C → Execução) com **trechos essenciais** e **comentários descritivos**. Sem recapitular vocabulário da linguagem.

---

## 1) Ponto de entrada e orquestração
O método `traduzir` conecta todas as fases. O `main` cuida de CLI e `--run` compila/roda o C.

```python
def traduzir(codigo_fonte: str) -> str:
    # 1) LÉXICO: quebra o texto em tokens
    lexico = AnalisadorLexico(codigo_fonte)
    tokens = lexico.tokenizar()

    # 2) PARSER: consome tokens e constrói a AST (árvore sintática)
    parser = AnalisadorSintatico(tokens)
    prog = parser.analisar_programa()

    # 3) SEMÂNTICA: valida declarações/uso de variáveis, tipos, índices etc.
    sem = AnalisadorSemantico()
    sem.analisar(prog)

    # 4) CODEGEN: percorre a AST + tabela de símbolos e produz código C
    gerador = GeradorCodigo(prog, sem)
    return gerador.gerar()

def executar_codigo_c(codigo_c: str, nome_arquivo: str = "saida.c"):
    # Gera um arquivo .c, compila com gcc e executa o binário
    with open(nome_arquivo, "w", encoding="utf-8") as f:
        f.write(codigo_c)
    subprocess.run(["gcc", nome_arquivo, "-o", "saida_exec"], check=True)
    resultado = subprocess.run(["./saida_exec"], capture_output=True, text=True)
    print(resultado.stdout)  # imprime stdout do programa em C
```

---

## 2) Léxico — `AnalisadorLexico.tokenizar()`
Responsável por **reconhecer padrões** (regex) e emitir tokens, pulando espaços e comentários comuns, mantendo comentários de documentação (`///`) como nós na AST.

```python
def tokenizar(self) -> List[Token]:
    tokens = []
    for mo in MASTER_RE.finditer(self.texto):  # percorre todo o fonte aplicando regex mestre
        tipo = mo.lastgroup       # nome do token reconhecido (ID, INT, STRING, ...)
        lex  = mo.group()         # lexema capturado (texto do token)
        pos  = mo.start()         # posição no texto (para mensagens de erro)

        if tipo in ("WHITESPACE", "COMMENT"):
            continue              # ignora espaços/linhas //comentário

        if tipo == "ID":
            k = KEYWORDS.get(lex) # se o lexema for palavra-chave, troca tipo -> TIPO_...
            tokens.append(Token(k or "ID", lex, pos))
            continue

        if tipo == "TRICOMMENT":
            # Comentário de 3 barras (///) vira nó Comentario na AST (documentação inline)
            texto = lex[3:].strip()
            tokens.append(Token("COMENTARIO", texto, pos))
            continue

        # Literais numéricos e string são preservados
        if tipo in ("INT", "FLOAT", "STRING"):
            tokens.append(Token(tipo, lex, pos))
            continue

        # Operadores, pontuação etc.
        tokens.append(Token(tipo, lex, pos))

    tokens.append(Token("EOF", None, len(self.texto)))  # sentinela de fim
    return tokens
```

**Destaques do léxico**  
- Palavras‑chave mapeadas para **tipos C** (ex.: `TIPO_INTEIRAO` → `int`).  
- `TRICOMMENT (///...)` vira **nó Comentario** (útil para gerar `/* ... */` em C).  
- `STRING` preserva escapes para tratamento especial no codegen (ver Seção 6).

---

## 3) AST — nós essenciais
Estruturas mínimas para representar **comandos** e **expressões**; variáveis suportam **vetores/matrizes** via `dims`.

```python
@dataclass
class DeclaracaoVar(Comando):
    tipo_base: str     # "int" | "float" | "double" | "char" (boolean mapeado para int)
    nome: str
    dims: List[int]    # [] escalar, [N] vetor, [M,N] matriz (1D/2D)
    inicial: Optional["Expr"]  # expressão de inicialização (escalar)

@dataclass
class AcessoArray(LValue):
    nome: str
    indices: List[Expr]  # permite 1D ou 2D: v[i] ou m[i][j]

@dataclass
class Mostra(Comando):
    expr: "Expr"  # aceita concatenação com '+'; vira printf com placeholders
```

**Por que isso importa?**  
O codegen precisa saber **tipo base** e **dimensões** para declarar arrays em C (`int v[3]; int m[2][2];`) e para montar `printf/scanf` corretos.

---

## 4) Parser — do token ao comando/expressão
O `comando()` decide **o que construir** (declaração, `sejoelho`, `praCada` etc.). `declaracao_var()` reconhece arrays. `_lvalue()` suporta acesso índice a índice.

```python
def comando(self) -> Comando:
    t = self.espiar().tipo
    if t in ("TIPO_INTEIRAO","TIPO_FLUTUANTE","TIPO_DOBRADURA","TIPO_BOOLEANO","TIPO_CHAR"):
        return self.declaracao_var()  # ex.: inteirao v[3];
    if t == "MOSTRA":    return self.comando_mostra()    # mostraAi(...);
    if t == "ENTRADA":   return self.comando_entrada()   # entradaAi(...);
    if t == "SE":        return self.comando_se()        # bloco if/else
    if t == "ENQUANTO":  return self.comando_enquanto()  # while (...){...}
    if t == "PARA":      return self.comando_para()      # for (...;...;...){...}
    if t == "LBRACE":    return self.bloco()             # { ... }
    if t == "COMENTARIO":# '///' vira nó Comentario
        texto = self.avancar().lexema or ""
        return Comentario(texto)
    # Sobrou: expressão/atribuição simples terminada por ';'
    expr = self.expressao()
    self.consumir("SEMI","Esperado ';' após expressão")
    return ExpressaoStmt(expr)

def declaracao_var(self) -> DeclaracaoVar:
    tipo_c = self._mapear_tipo(self.avancar())  # mapeia TIPO_... → "int"/"double"/...
    nome = self.consumir("ID","Esperado identificador").lexema

    dims = []
    while self.combinar("LBRACKET"):            # suporta v[10] e m[2][3]
        tam = int(self.consumir("INT","Esperado tamanho inteiro").lexema)
        self.consumir("RBRACKET","Esperado ']'")
        dims.append(tam)

    inicial = self.expressao() if self.combinar("ASSIGN") else None
    self.consumir("SEMI","Esperado ';' após declaração")
    return DeclaracaoVar(tipo_c, nome, dims, inicial)
```

**Hierarquia de expressão**: `logical_or → logical_and → igualdade → comparacao → termo (+/-) → fator (*//%) → unario → primario` (garante precedências corretas).  
**Atribuição em expressão**: em `primario`, `ID` pode virar `lvalue = expr` para permitir `x = y + 1` dentro de expressões.

---

## 5) Semântica — tabela de símbolos e validações
Garante **declaração prévia**, **dimensões coerentes** e **tipagem aproximada** (promoção `int → float → double` e regra de **string**).

```python
class AnalisadorSemantico:
    def __init__(self):
        self.tabela: Dict[str, Simbolo] = {}  # nome → (tipo_base, dims)

    def _obter(self, nome: str) -> Simbolo:
        if nome not in self.tabela:
            raise ErroSemantico(f"Variável '{nome}' usada antes da declaração")
        return self.tabela[nome]

    def _verificar_cmd(self, cmd: Comando) -> None:
        if isinstance(cmd, DeclaracaoVar):
            if cmd.nome in self.tabela:
                raise ErroSemantico("redeclaração")
            self.tabela[cmd.nome] = Simbolo(cmd.tipo_base, cmd.dims)
            if cmd.inicial is not None:
                self._tipo_expr(cmd.inicial)  # obriga check do inicializador
        elif isinstance(cmd, Atribuicao):
            # Se for array, exige TODOS os índices; se for escalar, nenhum
            if isinstance(cmd.alvo, AcessoArray):
                simb = self._obter(cmd.alvo.nome)
                if len(cmd.alvo.indices) != len(simb.dims):
                    raise ErroSemantico("Atribuição em array requer todos os índices")
            self._tipo_expr(cmd.expr)         # força verificação do lado direito
        # ... checa Se/Enquanto/Para, Mostra e Entrada de forma semelhante

    def _tipo_expr(self, e: Expr) -> str:
        # Regra de string: char[] → "string"; concatenação com '+' propaga "string"
        # Promoção numérica: int < float < double, char tratado como int
        # Atribuição retorna tipo do lado direito para encadear validação
        ...
```

**Regras chave**  
- `verdadeiro/falso` são modelados como **int (0/1)**.  
- `char[]` (strings) têm tratamento especial em **printf/scanf** (ver abaixo).  
- Índices excessivos/insuficientes em arrays **lançam erro semântico**.

---

## 6) Geração de C — `GeradorCodigo`
Emite **includes**, **declarações globais**, configura `locale`, e traduz cada comando da AST. Trata strings e E/S de forma segura.

```python
def gerar(self) -> str:
    self.linhas = [
        "#include <stdio.h>",     # printf / scanf
        "#include <string.h>",    # manipulação de strings se necessário
        "#include <locale.h>",    # acentuação/UTF‑8 no Windows
        "",
        "int main(void) {",
    ]

    # 1) Declara todas variáveis do programa no topo do main,
    #    respeitando base e dimensões (ex.: int v[3]; double m[2][2];)
    for nome, simb in self.sem.tabela.items():
        self.linhas.append("    " + self._decl_c(nome, simb) + ";")

    # 2) Habilita locale: ajuda na exibição de acentos no Windows
    self.linhas.append('    setlocale(LC_ALL, "");')

    # 3) Emite cada comando (if/while/for/mostra/entrada/etc.)
    for cmd in self.prog.comandos:
        self._emit_cmd(cmd, indent=1)

    self.linhas.append("    return 0;")
    self.linhas.append("}")
    return "\n".join(self.linhas)
```

### 6.1 `mostraAi` → `printf` com concatenação segura
Constrói **formato** + **argumentos**: strings viram `%s`; números usam `%d/%g`. Escapes são normalizados para evitar **quebras de linha acidentais** e **caracteres não escapados**.

```python
def _montar_printf(self, expr: Expr) -> Tuple[str, List[str]]:
    partes = self._flatten_concat(expr)  # "a" + x + "b" + y → ["a", x, "b", y]
    fmt_parts, args = [], []
    for p in partes:
        if isinstance(p, str):
            fmt_parts.append(p.replace('"','\\"').replace('%','%%'))  # protege aspas e %
        else:
            tipo = self._inferir_tipo_expr(p)
            fmt_parts.append("%s" if tipo == "string" else PRINTF_FMT.get(tipo,"%g"))
            args.append(self._emit_expr(p))
    return "".join(fmt_parts), args

def _escape_c_string(self, s: str) -> str:
    # Converte '\n','\t' etc. do VoidKnee para C e re‑escapa para literal
    s = s.replace('\\n','\n').replace('\\t','\t').replace('\\r','\r').replace('\\0','\0')
    s = s.replace('\\','\\\\').replace('"','\\"')
    s = s.replace('\n','\\n').replace('\t','\\t').replace('\r','\\r').replace('\0','\\0')
    return s
```

### 6.2 `entradaAi` → `scanf` segura (inclui **largura** para `char[N]`)
Lê em **escalares**, **elemento de array** ou em **string inteira** (`char[N]`), sempre validando o retorno de `scanf` para evitar **lixo de entrada**.

```python
def _emit_entrada(self, cmd: Entrada, indent: int) -> None:
    if isinstance(cmd.alvo, Variavel):
        simb = self.sem.tabela[cmd.alvo.nome]
        if len(simb.dims) == 0:
            # Escalar: usa spec do tipo (ex.: int → "%d") e valida
            spec = SCANF_FMT.get(simb.tipo_base, "%d")
            self.linhas.append(self._indent(indent) +
                f'if (scanf("{spec}", &{cmd.alvo.nome}) != 1) {{ fprintf(stderr, "Entrada inválida\\n"); return 1; }}')
        else:
            # String: char[N] sem índices → "%Ns" para evitar overflow e garantir '\0'
            if simb.tipo_base == "char" and len(simb.dims) == 1:
                largura = max(1, simb.dims[0] - 1)  # reserva 1 byte para terminador NUL
                self.linhas.append(self._indent(indent) +
                    f'if (scanf("%{largura}s", {cmd.alvo.nome}) != 1) {{ fprintf(stderr, "Entrada inválida\\n"); return 1; }}')
            else:
                raise RuntimeError("entradaAi: apenas char[N] completo é suportado")
    elif isinstance(cmd.alvo, AcessoArray):
        # Elemento específico de array (ex.: v[i] ou m[i][j])
        expr_c = f"{cmd.alvo.nome}" + "".join(f"[{self._emit_expr(idx)}]" for idx in cmd.alvo.indices)
        simb = self.sem.tabela[cmd.alvo.nome]
        spec = SCANF_FMT.get(simb.tipo_base, "%d")
        if simb.tipo_base == "char" and len(simb.dims) == len(cmd.alvo.indices):
            spec = SCANF_FMT["char"]  # leitura de um único char
        self.linhas.append(self._indent(indent) +
            f'if (scanf("{spec}", &({expr_c})) != 1) {{ fprintf(stderr, "Entrada inválida\\n"); return 1; }}')
```

### 6.3 `praCada` → `for (...) { ... }`
Traduz inicial/condição/passo respeitando **declaração/atribuição/expressão** na cabeça do laço.

```python
def _emit_cmd(self, cmd: Comando, indent: int) -> None:
    if isinstance(cmd, ParaCada):
        # p1 = inicial (pode vir de declaração, atribuição ou expr)
        p1 = ""
        if isinstance(cmd.inicial, DeclaracaoVar) and cmd.inicial.inicial and not cmd.inicial.dims:
            p1 = f"{cmd.inicial.nome} = {self._emit_expr(cmd.inicial.inicial)}"
        elif isinstance(cmd.inicial, Atribuicao):
            p1 = f"{self._emit_lvalue(cmd.inicial.alvo)} = {self._emit_expr(cmd.inicial.expr)}"
        elif isinstance(cmd.inicial, ExpressaoStmt):
            p1 = f"{self._emit_expr(cmd.inicial.expr)}"

        p2 = self._emit_expr(cmd.cond) if cmd.cond else ""  # condição

        # p3 = passo (atribuição ou expressão)
        if isinstance(cmd.passo, Atribuicao):
            p3 = f"{self._emit_lvalue(cmd.passo.alvo)} = {self._emit_expr(cmd.passo.expr)}"
        else:
            p3 = f"{self._emit_expr(cmd.passo.expr)}"

        self.linhas.append(self._indent(indent) + f"for ({p1}; {p2}; {p3}) " + "{")
        for c in cmd.corpo.comandos:
            self._emit_cmd(c, indent+1)
        self.linhas.append(self._indent(indent) + "}")
        return
```

---

## 7) Convenções e decisões de projeto
- **Boolean** da DSL → `int` em C (`0/1`).  
- **Strings**: representadas como `char[N]` e tratadas como `%s` em `printf`.  
- **Declarações no topo**: todas as variáveis aparecem no início do `main` para facilitar a geração.  
- **Erros amigáveis**: sintáticos com posição (`pos`), semânticos com mensagens claras (redeclaração, índices, uso antes da declaração).  
- **Locale** configurado em C para melhorar exibição de acentos em Windows (`setlocale(LC_ALL, "")`).

---


## Exemplos (gerando C)

### A) `sejoelho` / `outracoisa`

In [15]:
# Exemplo A — Condicional (sejoelho / outracoisa)
# Demonstra o uso do if/else da linguagem VoidKnee
# Entrada: um número inteiro
# Saída: "positivo" ou "negativo ou zero"

source = """
inteirao x;
mostraAi("Digite um número: ");
entradaAi(x);

sejoelho (x > 0) {
    mostraAi("positivo\n");
} outracoisa {
    mostraAi("negativo ou zero\n");
}
"""

c_code = gerar_c("v2_A_if", source)
print(c_code)


Gerado: C:\Users\user\Documents\Puc\Compiladores\desenvolvimento compilador\voidknee\src\out\v2_A_if.c
#include <stdio.h>
#include <string.h>
#include <locale.h>

int main(void) {
    int x;
    setlocale(LC_ALL, "");
    printf(u8"Digite um número: ");
    if (scanf("%d", &x) != 1) { fprintf(stderr, "Entrada inválida\n"); return 1; }
    if ((x > 0)) {
        printf(u8"positivo\n");
    }
    else {
        printf(u8"negativo ou zero\n");
    }
    return 0;
}


### B) `praCada`

In [16]:
# Exemplo B — praCada (for)
# Demonstra o laço "praCada" que equivale ao "for" em C
# Entrada: um limite inteiro
# Saída: imprime "i = 0", "i = 1", ..., até limite-1

source = """
inteirao limite;
mostraAi("Digite um limite: ");
entradaAi(limite);

inteirao i;
praCada (i = 0; i < limite; i = i + 1) {
    mostraAi("i = " + i + "\n");
}
"""

c_code = gerar_c("v2_B_for", source)
print(c_code)


Gerado: C:\Users\user\Documents\Puc\Compiladores\desenvolvimento compilador\voidknee\src\out\v2_B_for.c
#include <stdio.h>
#include <string.h>
#include <locale.h>

int main(void) {
    int limite;
    int i;
    setlocale(LC_ALL, "");
    printf(u8"Digite um limite: ");
    if (scanf("%d", &limite) != 1) { fprintf(stderr, "Entrada inválida\n"); return 1; }
    for (i = 0; (i < limite); (i = (i + 1))) {
        printf(u8"i = %d\n", i);
    }
    return 0;
}


### C) `enquantoDoi`

In [19]:
# Exemplo C — enquantoDoi (while)
# Demonstra o laço "enquantoDoi", equivalente ao "while" em C
# Inicializa x = 3 e imprime decrementando até 0

source = """
inteirao x;
x = 20;
enquantoDoi (x > 0) {
    mostraAi("x = " + x + "\n");
    x = x - 1;
}
"""

c_code = gerar_c("v2_C_while", source)
print(c_code)


Gerado: C:\Users\user\Documents\Puc\Compiladores\desenvolvimento compilador\voidknee\src\out\v2_C_while.c
#include <stdio.h>
#include <string.h>
#include <locale.h>

int main(void) {
    int x;
    setlocale(LC_ALL, "");
    (x = 20);
    while ((x > 0)) {
        printf(u8"x = %d\n", x);
        (x = (x - 1));
    }
    return 0;
}


### D) Tipos numéricos + booleano

In [20]:
# Exemplo D — Tipos numéricos + booleano
# Demonstra coerções automáticas e booleanos tratados como 0/1
# Mostra operações com int, float, double e bool

source = """
inteirao    a; a = 2;
flutuante   b; b = 3.5;
dobradura   c; c = 10.0;
verdadeQueDoi ok; ok = verdadeiro;

dobradura r1; r1 = a + b;
dobradura r2; r2 = c / a;
inteirao  r3; r3 = ok + 3;       // 1 + 3 = 4
inteirao  r4; r4 = (ok && falso);// 0

mostraAi("r1=" + r1 + ", r2=" + r2 + ", r3=" + r3 + ", r4=" + r4 + "\n");
"""

c_code = gerar_c("v2_D_tipos", source)
print(c_code)


Gerado: C:\Users\user\Documents\Puc\Compiladores\desenvolvimento compilador\voidknee\src\out\v2_D_tipos.c
#include <stdio.h>
#include <string.h>
#include <locale.h>

int main(void) {
    int a;
    float b;
    double c;
    int ok;
    double r1;
    double r2;
    int r3;
    int r4;
    setlocale(LC_ALL, "");
    (a = 2);
    (b = 3.5);
    (c = 10.0);
    (ok = 1);
    (r1 = (a + b));
    (r2 = (c / a));
    (r3 = (ok + 3));
    (r4 = (ok && 0));
    printf(u8"r1=%lf, r2=%lf, r3=%d, r4=%d\n", r1, r2, r3, r4);
    return 0;
}


### E) Vetores e Matrizes

In [24]:
# Exemplo E — Vetores e Matrizes
# Demonstra o uso de arrays e matrizes com loops aninhados
# Calcula soma de dois vetores e monta matriz identidade (bem formatada na saída)

source = """
inteirao i;
inteirao j;
inteirao v1[3]; inteirao v2[3]; inteirao soma[3];
praCada (i = 0; i < 3; i = i + 1) {
    v1[i] = i;
    v2[i] = i * 2;
    soma[i] = v1[i] + v2[i];
}

inteirao m[3][3];
praCada (i = 0; i < 3; i = i + 1) {
    praCada (j = 0; j < 3; j = j + 1) {
        m[i][j] = (i == j);
    }
}

mostraAi("Soma dos vetores: [" + soma[0] + ", " + soma[1] + ", " + soma[2] + "]\\n");
mostraAi("Matriz identidade 3x3:\\n");

praCada (i = 0; i < 3; i = i + 1) {
    praCada (j = 0; j < 3; j = j + 1) {
        mostraAi(m[i][j] + " ");
    }
    mostraAi("\\n");
}
"""

c_code = gerar_c("v2_E_arrays", source)
print(c_code)


Gerado: C:\Users\user\Documents\Puc\Compiladores\desenvolvimento compilador\voidknee\src\out\v2_E_arrays.c
#include <stdio.h>
#include <string.h>
#include <locale.h>

int main(void) {
    int i;
    int j;
    int v1[3];
    int v2[3];
    int soma[3];
    int m[3][3];
    setlocale(LC_ALL, "");
    for (i = 0; (i < 3); (i = (i + 1))) {
        (v1[i] = i);
        (v2[i] = (i * 2));
        (soma[i] = (v1[i] + v2[i]));
    }
    for (i = 0; (i < 3); (i = (i + 1))) {
        for (j = 0; (j < 3); (j = (j + 1))) {
            (m[i][j] = (i == j));
        }
    }
    printf(u8"Soma dos vetores: [%d, %d, %d]\n", soma[0], soma[1], soma[2]);
    printf(u8"Matriz identidade 3x3:\n");
    for (i = 0; (i < 3); (i = (i + 1))) {
        for (j = 0; (j < 3); (j = (j + 1))) {
            printf(u8"%d ", m[i][j]);
        }
        printf(u8"\n");
    }
    return 0;
}


### F) Strings e `dorzinha`

In [22]:
# Exemplo F — Strings e char
# Demonstra o uso de "dorzona" (string) e "dorzinha" (char)
# Lê nome e uma letra, depois imprime ambos

source = """
dorzona nome[16];
mostraAi("Seu nome? ");
entradaAi(nome);

mostraAi("Olá, " + nome + "!\n");

dorzinha letra;
mostraAi("Digite uma letra qualquer: ");
entradaAi(letra);

mostraAi("Você digitou: " + letra + "\n");
"""

c_code = gerar_c("v2_F_strings", source)
print(c_code)


Gerado: C:\Users\user\Documents\Puc\Compiladores\desenvolvimento compilador\voidknee\src\out\v2_F_strings.c
#include <stdio.h>
#include <string.h>
#include <locale.h>

int main(void) {
    char nome[16];
    char letra;
    setlocale(LC_ALL, "");
    printf(u8"Seu nome? ");
    if (scanf("%15s", nome) != 1) { fprintf(stderr, "Entrada inválida\n"); return 1; }
    printf(u8"Olá, %s!\n", nome);
    printf(u8"Digite uma letra qualquer: ");
    if (scanf(" %c", &letra) != 1) { fprintf(stderr, "Entrada inválida\n"); return 1; }
    printf(u8"Você digitou: %c\n", letra);
    return 0;
}


## 👏 Sessão de Palmas

> *aplausos ecoam pela sala*

👏👏👏👏👏  👏👏👏👏👏  
🙌🎉🙌🎉🙌  🙌🎉🙌🎉🙌  
✨👏✨👏✨  ✨👏✨👏✨  
👏👏👏👏👏  👏👏👏👏👏  
🙌🎉🙌🎉🙌  🙌🎉🙌🎉🙌  
✨👏✨👏✨  ✨👏✨👏✨  

**Obrigado! equipe VoidKnee!** 🦵🚀
